<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/QWEN-2511-LORA-TRAINING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen Image Edit 2511 LoRA training (A100 / L4)

Trains a LoRA for **Qwen-Image-Edit-2511** with [musubi-tuner](https://github.com/kohya-ss/musubi-tuner). The finished LoRA
is saved to `MyDrive/ComfyUI/loras_qwen`, so it shows up in the LoRA list of both Qwen edit notebooks.

| LoRA type | What you give it | GPU |
|---|---|---|
| **Character** | 15–30 pictures of one character (+ trigger word) | **A100** (1024 px) or **L4** (768 px, slower) |
| **Edit effect** | 10–30 **pairs**: a *Before* picture, the *After* picture, and an instruction | **A100** |

**With the website**
1. **Runtime → Change runtime type → A100 GPU** (or L4 for character LoRAs). Don't run another GPU notebook at the same time.
2. Run **cell 1** (setup + ~58 GB of model downloads, about 10–15 minutes; Colab asks for Google Drive access) and
   **cell 2** (prints a `READY:` URL).
3. On the website open **LoRA Training**, paste the URL (the tab switches to Qwen 2511), name the LoRA, pick the
   **LoRA type**, add pictures (and *Before* pictures for an edit effect), check the captions and press **Start training**.
4. Training takes roughly 1–3 hours on an A100 (longer on an L4); the website shows the progress. Keep this Colab tab open.

Progress is saved every 200 steps: if Colab disconnects, run cells 1–2 again and press Start training with the same
name to continue. **Cell 4** shows the progress here too. **Without the website:** cell 3 trains a character LoRA.

Pictures are kept in `MyDrive/ComfyUI/training_qwen/<name>/` (`images` = training pictures + captions,
`control` = Before pictures). Only train on pictures you have the rights to use, and only make a LoRA of a real person
with that person's consent.

In [ ]:
# Cell 1: setup — Google Drive, musubi-tuner, the Qwen-Image-Edit-2511 models, and the shared training code
%cd /content
import os, re, io, json, glob, time, shutil, signal, hashlib, threading, subprocess, collections

from google.colab import drive
drive.mount("/content/drive")  # Colab asks you to allow access

DRIVE_ROOT = "/content/drive/MyDrive/ComfyUI"
TRAIN_ROOT = f"{DRIVE_ROOT}/training_qwen"   # <name>/images (+ captions), <name>/control (before pictures), <name>/output
LORAS_QWEN = f"{DRIVE_ROOT}/loras_qwen"      # finished LoRAs go here (both Qwen edit notebooks read this folder)
os.makedirs(TRAIN_ROOT, exist_ok=True)
os.makedirs(LORAS_QWEN, exist_ok=True)

# ---------------------------------------------------------------- GPU check
# Qwen-Image-Edit-2511 is a ~20B model: training needs a bf16 GPU with 20+ GB.
#   L4 (22.5 GB):  Character LoRAs, at 768 px, with part of the model kept in RAM (block swap) — slower
#   A100 (40 GB):  Character and Edit-effect LoRAs at 1024 px
try:
    _gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True).stdout.strip().split("\n")[0]
except FileNotFoundError:  # nvidia-smi only exists when a GPU is attached
    _gpu = ""
if _gpu.count(",") < 2:
    raise RuntimeError(
        "This Colab runtime has no GPU, and training needs one.\n"
        "Fix: Runtime → Change runtime type → A100 (or L4) GPU → Save, then run the cells again.")
_name, _mib, _cap = [x.strip() for x in _gpu.rsplit(",", 2)]
GPU = {"name": _name, "vram_gib": round(int(_mib) / 1024, 1), "compute_cap": _cap}
if float(_cap or 0) < 8.0 or GPU["vram_gib"] < 20:
    raise RuntimeError(
        f"This GPU ({GPU['name']}, {GPU['vram_gib']} GB) is too small for Qwen-Image-Edit-2511 training.\n"
        "Pick Runtime → Change runtime type → A100 GPU (best) or L4 GPU (character LoRAs only), then run the cells again.")
if GPU["vram_gib"] >= 70:      # A100 80 GB / H100
    PROFILE = {"label": "big GPU", "edit_ok": True, "resolution": 1024, "swap_character": 0, "swap_edit": 0, "fp8_vl": False}
elif GPU["vram_gib"] >= 36:    # A100 40 GB
    PROFILE = {"label": "A100", "edit_ok": True, "resolution": 1024, "swap_character": 4, "swap_edit": 14, "fp8_vl": False}
else:                          # L4 22.5 GB
    PROFILE = {"label": "L4", "edit_ok": False, "resolution": 768, "swap_character": 30, "swap_edit": 0, "fp8_vl": True}
GPU["edit_ok"] = PROFILE["edit_ok"]
GPU["mode"] = "improved"
GPU["mode_label"] = (f"{PROFILE['label']}: character and edit-effect LoRAs at 1024 px" if PROFILE["edit_ok"]
                     else "L4: character LoRAs at 768 px (part of the model is kept in RAM, so it's slower); "
                          "edit-effect LoRAs need an A100")
print(f"GPU: {GPU['name']} ({GPU['vram_gib']} GB) -> {GPU['mode_label']}")

# ---------------------------------------------------------------- install musubi-tuner (pinned) and download the models
MUSUBI = "/content/musubi-tuner"
MUSUBI_COMMIT = "f8a1b03"
if not os.path.exists(MUSUBI):
    !git clone -q https://github.com/kohya-ss/musubi-tuner {MUSUBI}
!cd {MUSUBI} && git fetch -q origin && git checkout -q {MUSUBI_COMMIT}
!cd {MUSUBI} && pip install -q -e .
!pip install -q fastapi uvicorn nest_asyncio
!apt install -qqy aria2 > /dev/null
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# Training needs the full bf16 model files (the fp8 ones can't be trained); musubi converts them to fp8 while loading.
MODELS = "/content/qwen_models"
DIT = f"{MODELS}/qwen_image_edit_2511_bf16.safetensors"   # ~41 GB
TEXT_ENCODER = f"{MODELS}/qwen_2.5_vl_7b.safetensors"     # ~16.6 GB (also used for Auto-caption)
VAE = f"{MODELS}/qwen_image_vae.safetensors"
DOWNLOADS = [
    ("https://huggingface.co/Comfy-Org/Qwen-Image-Edit/resolve/main/split_files/diffusion_models/qwen_image_edit_2511_bf16.safetensors", DIT),
    ("https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b.safetensors", TEXT_ENCODER),
    ("https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors", VAE),
]
os.makedirs(MODELS, exist_ok=True)
for url, path in DOWNLOADS:
    if not os.path.exists(path) or os.path.exists(path + ".aria2"):
        print(f"Downloading {os.path.basename(path)} (the model is big: about 10 minutes in total)...")
        subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M", url,
                        "-d", MODELS, "-o", os.path.basename(path)], check=True)

# ---------------------------------------------------------------- shared training code
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
KINDS = ("character", "edit")

DEFAULTS = {
    "trigger_word": "",
    "class_word": "woman",
    "add_trigger_to_captions": True,
    "max_train_steps": 1200 if PROFILE["edit_ok"] else 1000,
    "num_repeats": 1,
    "train_batch_size": 1,
    "resolution": PROFILE["resolution"],
    "network_dim": 16,
    "network_alpha": 16,
    "learning_rate": 1e-4,
    "seed": 42,
    "save_every_n_steps": 200,
}


def clean_name(name):
    return re.sub(r"[^\w\-]+", "_", (name or "").strip()).strip("_")


def clean_file(file):
    file = os.path.basename(file or "")
    stem, ext = os.path.splitext(file)
    stem = re.sub(r"[^\w\-. ]+", "_", stem).strip(" .") or "image"
    return stem + ext.lower()


def images_dir(name):
    n = clean_name(name)
    if not n:
        raise ValueError("Give the LoRA a name first")
    return f"{TRAIN_ROOT}/{n}/images"


def control_dir(name):
    return f"{TRAIN_ROOT}/{clean_name(name)}/control"


def output_dir(name):
    return f"{TRAIN_ROOT}/{clean_name(name)}/output"


def list_images(name):
    d = images_dir(name)
    return sorted(f for f in os.listdir(d) if f.lower().endswith(IMAGE_EXTS)) if os.path.isdir(d) else []


def control_for(name, file):
    """The 'before' picture saved for a training image (edit-effect LoRAs), or None."""
    stem = os.path.splitext(clean_file(file))[0]
    d = control_dir(name)
    if os.path.isdir(d):
        for f in os.listdir(d):
            if os.path.splitext(f)[0] == stem and f.lower().endswith(IMAGE_EXTS):
                return f
    return None


def caption_path(name, file):
    return os.path.join(images_dir(name), os.path.splitext(clean_file(file))[0] + ".txt")


def read_caption(name, file):
    p = caption_path(name, file)
    return open(p, encoding="utf-8").read().strip() if os.path.exists(p) else ""


def write_caption(name, file, caption):
    with open(caption_path(name, file), "w", encoding="utf-8") as fh:
        fh.write((caption or "").strip())


def split_tags(caption):
    return [t.strip() for t in (caption or "").split(",") if t.strip()]


def get_kind(name):
    try:
        return json.load(open(f"{TRAIN_ROOT}/{clean_name(name)}/meta.json")).get("kind", "character")
    except Exception:
        return "character"


def set_kind(name, kind):
    if kind not in KINDS:
        raise ValueError("LoRA type must be 'character' or 'edit'")
    os.makedirs(f"{TRAIN_ROOT}/{clean_name(name)}", exist_ok=True)
    json.dump({"kind": kind}, open(f"{TRAIN_ROOT}/{clean_name(name)}/meta.json", "w"))


def list_datasets():
    out = []
    for n in sorted(os.listdir(TRAIN_ROOT)):
        if os.path.isdir(os.path.join(TRAIN_ROOT, n, "images")):
            out.append({"name": n, "images": len(list_images(n)), "kind": get_kind(n),
                        "has_lora": os.path.exists(f"{LORAS_QWEN}/{n}.safetensors")})
    return out


def saved_progress(name):
    """Latest saved step of an unfinished/finished run with this name (None if there's none)."""
    states = glob.glob(f"{output_dir(name)}/{clean_name(name)}-step*-state")
    steps = [int(re.search(r"-step(\d+)-state", p).group(1)) for p in states]
    return max(steps) if steps else None


def start_fresh(name):
    """Move the previous run aside so the next training starts from zero (nothing is deleted)."""
    n = clean_name(name)
    if busy() and STATE["name"] == n:
        raise RuntimeError(f"'{n}' is training right now — stop it first")
    base = f"{TRAIN_ROOT}/{n}"
    v = 1
    while os.path.exists(f"{base}/output_v{v}") or os.path.exists(f"{LORAS_QWEN}/{n}_v{v}.safetensors"):
        v += 1
    moved = []
    if os.path.isdir(f"{base}/output") and os.listdir(f"{base}/output"):
        os.rename(f"{base}/output", f"{base}/output_v{v}")
        moved.append(f"training_qwen/{n}/output → output_v{v}")
    if os.path.exists(f"{LORAS_QWEN}/{n}.safetensors"):
        os.rename(f"{LORAS_QWEN}/{n}.safetensors", f"{LORAS_QWEN}/{n}_v{v}.safetensors")
        moved.append(f"loras_qwen/{n}.safetensors → {n}_v{v}.safetensors")
    return {"moved": moved, "version": v if moved else None}


# ---- one background task at a time (training or auto-captioning), shared with the website
STATE = {"status": "idle", "task": None, "name": None, "phase": "", "step": 0, "total": 0,
         "elapsed": "", "eta": "", "error": None, "result": None, "started": None,
         "log": collections.deque(maxlen=60)}
_PROC = {"p": None, "stop": False}
PROGRESS_RE = re.compile(r"steps:\s*\d+%.*?\|\s*(\d+)/(\d+)\s*\[([^<\]]*)<([^,\]]*)")


def status_payload():
    s = {k: v for k, v in STATE.items() if k != "log"}
    s["log"] = list(STATE["log"])[-25:]
    return s


def _log(line):
    line = line.rstrip()
    if line:
        STATE["log"].append(line)


def _run(cmd, cwd):
    """Run a command, stream its output into STATE (the training progress bar updates step/eta)."""
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0, env=env,
                         start_new_session=True)  # stopping a notebook cell must not stop training
    _PROC["p"] = p
    buf = b""
    while True:
        chunk = p.stdout.read(4096)
        if not chunk:
            break
        buf += chunk
        parts = re.split(rb"[\r\n]", buf)
        buf = parts.pop()
        for raw in parts:
            line = raw.decode("utf-8", "replace").strip()
            if not line:
                continue
            m = PROGRESS_RE.search(line)
            if m:
                STATE.update(step=int(m.group(1)), total=int(m.group(2)),
                             elapsed=m.group(3).strip(), eta=m.group(4).strip(), phase="Training")
            else:
                _log(line)
    rc = p.wait()
    _PROC["p"] = None
    return rc


def busy():
    return STATE["status"] == "running"


def _begin(task, name, phase):
    if busy():
        raise RuntimeError(f"Busy: {STATE['task']} of '{STATE['name']}' is still running")
    STATE["log"].clear()
    STATE.update(status="running", task=task, name=clean_name(name), phase=phase, step=0, total=0,
                 elapsed="", eta="", error=None, result=None, started=time.time())
    _PROC["stop"] = False


def _finish(error=None, result=None):
    if _PROC["stop"]:
        STATE.update(status="stopped", phase="Stopped")
    elif error:
        STATE.update(status="error", error=str(error), phase="Failed")
    else:
        STATE.update(status="done", result=result, phase="Done")


def stop_task():
    _PROC["stop"] = True
    p = _PROC["p"]
    if p and p.poll() is None:
        try:
            os.killpg(p.pid, signal.SIGTERM)  # the trainer and its worker processes
        except Exception:
            p.terminate()


def prepare_dataset(name, s, kind):
    """Copy the training pictures (+ 'before' pictures) to the runtime and write musubi's dataset config.

    character: each picture is paired with a DIFFERENT picture of the same character as Picture 1, so the LoRA
               learns the character (and the trigger word) and to carry it over from a reference picture.
    edit:      each 'after' picture is paired with its own 'before' picture; the caption is the instruction."""
    imgs = list_images(name)
    local = f"/content/dataset/{clean_name(name)}"
    shutil.rmtree(local, ignore_errors=True)
    os.makedirs(f"{local}/images")
    os.makedirs(f"{local}/control")
    trigger, cls = s["trigger_word"].strip(), s["class_word"].strip()
    pairs = []
    if kind == "edit":
        for f in imgs:
            c = control_for(name, f)
            if c:
                pairs.append((f, c))
        missing = len(imgs) - len(pairs)
        if len(pairs) < 3:
            raise RuntimeError(f"An edit-effect LoRA needs 'before' pictures: only {len(pairs)} of {len(imgs)} "
                               f"training pictures have one. Add a Before picture to each (at least 3, ideally 10-30).")
        if missing:
            _log(f"{missing} picture(s) without a 'before' picture are left out")
    else:
        if len(imgs) < 4:
            raise RuntimeError(f"Add more pictures of '{name}' first — at least 4 (15-30 works best)")
        for i, f in enumerate(imgs):
            pairs.append((f, None))
    for i, (f, c) in enumerate(pairs):
        stem = os.path.splitext(f)[0]
        shutil.copy2(os.path.join(images_dir(name), f), f"{local}/images/{f}")
        if kind == "edit":
            src = os.path.join(control_dir(name), c)
        else:  # another picture of the same character, as the reference
            src = os.path.join(images_dir(name), imgs[(i + max(1, len(imgs) // 2)) % len(imgs)])
        shutil.copy2(src, f"{local}/control/{stem}{os.path.splitext(src)[1].lower()}")
        caption = read_caption(name, f)
        if kind == "character":
            # "<trigger> <class> from Picture 1, <description>": ties the trigger word to the character and to
            # the reference picture (strip a trigger word already at the start, e.g. from Auto-caption)
            who = " ".join(w for w in (trigger, cls) if w) or "the person"
            rest = caption
            if trigger and rest.lower().startswith(trigger.lower()):
                rest = rest[len(trigger):].lstrip(" ,")
            if s["add_trigger_to_captions"] or not caption:
                caption = f"{who} from Picture 1" + (f", {rest}" if rest else "")
        elif not caption:
            raise RuntimeError(f"'{f}' has no instruction — write what changes from the Before picture "
                               f"(e.g. \"turn this photo into a comic-book drawing\")")
        with open(f"{local}/images/{stem}.txt", "w", encoding="utf-8") as fh:
            fh.write(caption)
    res = s["resolution"]
    toml_path = "/content/dataset/dataset.toml"
    with open(toml_path, "w") as fh:
        fh.write(f'''[general]
resolution = [{res}, {res}]
caption_extension = ".txt"
batch_size = 1
enable_bucket = true
bucket_no_upscale = false

[[datasets]]
image_directory = "{local}/images"
control_directory = "{local}/control"
control_resolution = [{res}, {res}]
cache_directory = "{local}/cache"
num_repeats = {s["num_repeats"]}
''')
    return toml_path, len(pairs)


def build_train_args(name, s, kind, toml_path):
    n, out = clean_name(name), output_dir(name)
    os.makedirs(out, exist_ok=True)
    states = sorted(glob.glob(f"{out}/{n}-step*-state"), key=lambda p: int(re.search(r"-step(\d+)-state", p).group(1)))
    saved_path = f"{out}/run_settings.json"
    saved = json.load(open(saved_path)) if states and os.path.exists(saved_path) else {}
    swap = PROFILE["swap_edit"] if kind == "edit" else PROFILE["swap_character"]
    if s["resolution"] > PROFILE["resolution"]:
        swap = max(swap, 40)  # more resolution than this GPU's default: keep more of the model in RAM
    args = [
        "accelerate", "launch", "--num_cpu_threads_per_process", "1", "--mixed_precision", "bf16",
        "src/musubi_tuner/qwen_image_train_network.py",
        "--dit", DIT, "--vae", VAE, "--text_encoder", TEXT_ENCODER, "--model_version", "edit-2511",
        "--dataset_config", toml_path, "--sdpa", "--mixed_precision", "bf16",
        "--fp8_base", "--fp8_scaled", *(["--blocks_to_swap", str(swap)] if swap else []),
        "--timestep_sampling", "shift", "--weighting_scheme", "none", "--discrete_flow_shift", "2.2",
        "--optimizer_type", "adamw8bit", "--learning_rate", str(s["learning_rate"]),
        "--lr_scheduler", "cosine", "--lr_warmup_steps", str(max(1, s["max_train_steps"] // 20)),
        "--gradient_checkpointing", "--max_data_loader_n_workers", "2", "--persistent_data_loader_workers",
        "--network_module", "networks.lora_qwen_image",
        "--network_dim", str(s["network_dim"]), "--network_alpha", str(s["network_alpha"]),
        "--max_train_steps", str(s["max_train_steps"]), "--seed", str(s["seed"]),
        "--save_every_n_steps", str(s["save_every_n_steps"]), "--save_state",
        "--save_last_n_steps_state", str(s["save_every_n_steps"] * 2),
        "--output_dir", out, "--output_name", n,
    ]
    # Settings that must match to continue a saved run
    must_match = {"kind": kind, **{k: s[k] for k in ("network_dim", "network_alpha", "resolution", "max_train_steps")}}
    if saved:
        diff = [f"{k}: {saved.get(k)} → {v}" for k, v in must_match.items() if saved.get(k) != v]
        if diff:
            raise RuntimeError(f"'{n}' already has saved progress with different settings ({'; '.join(diff)}). "
                               f"Press 'Start fresh' on the website (or use a new LoRA name) to train with the new "
                               f"settings, or switch back to the old settings to continue.")
    with open(saved_path, "w") as fh:
        json.dump(must_match, fh)
    if states:
        args += ["--resume", states[-1]]
        _log(f"Resuming from {os.path.basename(states[-1])}")
    return args


def merge_settings(settings):
    s = dict(DEFAULTS)
    for k, v in (settings or {}).items():
        if k in DEFAULTS and v is not None and v != "":
            t = type(DEFAULTS[k])
            if t is bool:
                s[k] = v if isinstance(v, bool) else str(v).lower() in ("1", "true", "yes")
            elif t is int:
                s[k] = int(float(v))
            else:
                s[k] = t(v)
    s["train_batch_size"] = 1
    s["resolution"] = min(max(512, s["resolution"] // 64 * 64), 1024)
    s["max_train_steps"] = max(50, s["max_train_steps"])
    s["save_every_n_steps"] = max(50, s["save_every_n_steps"])
    s["num_repeats"] = max(1, s["num_repeats"])
    return s


def _train_worker(name, s, kind):
    try:
        if kind == "edit" and not PROFILE["edit_ok"]:
            raise RuntimeError("Edit-effect LoRAs need an A100 (40 GB): Runtime → Change runtime type → A100, "
                               "then run cells 1-2 again. Character LoRAs work on this L4.")
        STATE["phase"] = "Preparing pictures"
        toml_path, n = prepare_dataset(name, s, kind)
        _log(f"{kind} LoRA: {n} pictures, {s['max_train_steps']} steps, {s['resolution']} px, GPU {GPU['name']}")
        STATE["phase"] = "Caching pictures (the first time takes a few minutes)"
        if _run(["python", "src/musubi_tuner/qwen_image_cache_latents.py", "--dataset_config", toml_path,
                 "--vae", VAE, "--model_version", "edit-2511"], MUSUBI) != 0:
            raise RuntimeError("Caching the pictures failed — see the log")
        if _PROC["stop"]:
            return _finish()
        STATE["phase"] = "Reading the captions with the text encoder"
        if _run(["python", "src/musubi_tuner/qwen_image_cache_text_encoder_outputs.py", "--dataset_config", toml_path,
                 "--text_encoder", TEXT_ENCODER, "--batch_size", "1", "--model_version", "edit-2511",
                 *(["--fp8_vl"] if PROFILE["fp8_vl"] else [])], MUSUBI) != 0:
            raise RuntimeError("Reading the captions failed — see the log")
        if _PROC["stop"]:
            return _finish()
        STATE.update(phase="Loading the model (several minutes: it's 41 GB)", total=s["max_train_steps"])
        rc = _run(build_train_args(name, s, kind, toml_path), MUSUBI)
        final = f"{output_dir(name)}/{clean_name(name)}.safetensors"
        if rc == 0 and os.path.exists(final):
            dest = f"{LORAS_QWEN}/{clean_name(name)}.safetensors"
            shutil.copy2(final, dest)
            _finish(result={"lora": os.path.basename(dest), "path": dest})
        else:
            tail = " | ".join(list(STATE["log"])[-3:])
            hint = " (out of GPU memory: try Resolution 768)" if "out of memory" in tail.lower() else ""
            _finish(error=f"Training failed (exit code {rc}){hint}. Last output: {tail}")
    except Exception as e:
        _finish(error=e)


def start_training(name, settings=None):
    s = merge_settings(settings)
    kind = get_kind(name)
    _begin("training", name, "Starting")
    STATE["settings"] = s
    threading.Thread(target=_train_worker, args=(name, s, kind), daemon=True).start()
    return s


CAPTION_PROMPT = ("Describe this image in one or two sentences for an image-editing dataset: what the person is doing, "
                  "their pose, clothing, the setting, framing and lighting. Do not describe the face, hair colour or "
                  "eye colour, and do not guess who it is.")


def _autotag_worker(name, mode, trigger):
    try:
        imgs = list_images(name)
        if not imgs:
            raise RuntimeError("No pictures to caption")
        work = f"/content/autocaption/{clean_name(name)}"
        shutil.rmtree(work, ignore_errors=True)
        os.makedirs(work)
        for f in imgs:
            shutil.copy2(os.path.join(images_dir(name), f), os.path.join(work, f))
        STATE["phase"] = "Auto-captioning with Qwen2.5-VL (loading takes a minute)"
        rc = _run(["python", "src/musubi_tuner/caption_images_by_qwen_vl.py", "--image_dir", work,
                   "--model_path", TEXT_ENCODER, "--output_format", "text", "--max_new_tokens", "160",
                   "--prompt", CAPTION_PROMPT, *(["--fp8_vl"] if PROFILE["fp8_vl"] else [])], MUSUBI)
        if rc != 0:
            raise RuntimeError("The captioner failed — see the log")
        changed = 0
        for f in imgs:
            p = os.path.join(work, os.path.splitext(f)[0] + ".txt")
            if not os.path.exists(p):
                continue
            new = " ".join(open(p, encoding="utf-8").read().split())
            old = read_caption(name, f)
            caption = f"{old}, {new}" if mode == "append" and old else new
            if trigger and trigger.lower() not in caption.lower():
                caption = f"{trigger}, {caption}"
            write_caption(name, f, caption)
            changed += 1
        _finish(result={"tagged": changed})
    except Exception as e:
        _finish(error=e)


def start_autotag(name, threshold=0.35, mode="replace", trigger="", exclude=""):
    """Natural-language captions with Qwen2.5-VL (threshold / exclude are for the SDXL tagger; not used here)."""
    _begin("auto-tagging", name, "Starting the captioner")
    threading.Thread(target=_autotag_worker, args=(name, mode, (trigger or "").strip()), daemon=True).start()


print("Setup done. Run cell 2 to use the website, or cell 3 to train without it.")

In [ ]:
# Cell 2: training server for the website (LoRA Training tab → it switches to Qwen 2511 by itself). Paste the READY: URL there.
# This cell finishes after printing the URL; the server keeps running in the background.
if "clean_name" not in globals():
    raise RuntimeError("Run cell 1 (setup) first, then run this cell again.")

import base64, io, re, subprocess, threading, time
import nest_asyncio, uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel
from PIL import Image, ImageOps

nest_asyncio.apply()
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"])


def fail(msg, code=400):
    raise HTTPException(status_code=code, detail=str(msg))


def dataset_payload(name):
    return {"name": clean_name(name), "kind": get_kind(name),
            "images": [{"file": f, "caption": read_caption(name, f), "control": control_for(name, f)}
                       for f in list_images(name)],
            "saved_step": saved_progress(name),
            "has_lora": os.path.exists(f"{LORAS_QWEN}/{clean_name(name)}.safetensors")}


@app.get("/")
def root():
    return {"status": "ok", "mode": "lora_training"}


@app.get("/training/info")
def info():
    return {"mode": "lora_training", "trainer": "qwen_2511", "defaults": DEFAULTS, "datasets": list_datasets(),
            "status": status_payload(), "gpu": GPU, "kinds": list(KINDS)}


@app.get("/training/status")
def status():
    return status_payload()


@app.get("/training/dataset/{name}")
def get_dataset(name: str):
    try:
        return dataset_payload(name)
    except ValueError as e:
        fail(e)


class Upload(BaseModel):
    filename: str
    data: str  # base64 (data: URL prefix allowed)
    caption: str | None = None


@app.post("/training/dataset/{name}/upload")
def upload(name: str, req: Upload):
    try:
        d = images_dir(name)
    except ValueError as e:
        fail(e)
    file = clean_file(req.filename)
    if not file.lower().endswith(IMAGE_EXTS):
        fail(f"{req.filename}: only jpg, png, webp and bmp images are supported")
    raw = base64.b64decode(re.sub(r"^data:[^,]*,", "", req.data))
    try:
        Image.open(io.BytesIO(raw)).verify()
    except Exception:
        fail(f"{req.filename} is not a readable image")
    os.makedirs(d, exist_ok=True)
    stem, ext = os.path.splitext(file)
    i = 1
    while os.path.exists(os.path.join(d, file)):  # never overwrite: photo.jpg -> photo-1.jpg
        file, i = f"{stem}-{i}{ext}", i + 1
    with open(os.path.join(d, file), "wb") as fh:
        fh.write(raw)
    if req.caption is not None:
        write_caption(name, file, req.caption)
    return {"file": file, "caption": read_caption(name, file)}


class Caption(BaseModel):
    file: str
    caption: str


@app.put("/training/dataset/{name}/caption")
def set_caption(name: str, req: Caption):
    if clean_file(req.file) not in list_images(name):
        fail("Image not found", 404)
    write_caption(name, req.file, req.caption)
    return {"file": req.file, "caption": read_caption(name, req.file)}


class Bulk(BaseModel):
    action: str  # "add" (to the start) or "remove"
    tag: str


@app.post("/training/dataset/{name}/bulk")
def bulk(name: str, req: Bulk):
    tag = req.tag.strip().strip(",").strip()
    if not tag:
        fail("Type a tag first")
    for f in list_images(name):
        tags = [t for t in split_tags(read_caption(name, f)) if t.lower() != tag.lower()]
        if req.action == "add":
            tags.insert(0, tag)
        write_caption(name, f, ", ".join(tags))
    return dataset_payload(name)


@app.delete("/training/dataset/{name}/image/{file}")
def delete_image(name: str, file: str):
    file = clean_file(file)
    if file not in list_images(name):
        fail("Image not found", 404)
    os.remove(os.path.join(images_dir(name), file))
    if os.path.exists(caption_path(name, file)):
        os.remove(caption_path(name, file))
    c = control_for(name, file)
    if c:
        os.remove(os.path.join(control_dir(name), c))
    return dataset_payload(name)


@app.get("/training/dataset/{name}/file/{file}")
def full_file(name: str, file: str):
    """The original image (for the website's cloud backup of the dataset)."""
    file = clean_file(file)
    src = os.path.join(images_dir(name), file)
    if file not in list_images(name) or not os.path.exists(src):
        fail("Image not found", 404)
    ext = os.path.splitext(file)[1].lower().lstrip(".")
    media = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png", "webp": "image/webp", "bmp": "image/bmp"}.get(ext, "application/octet-stream")
    return Response(open(src, "rb").read(), media_type=media)


THUMBS = "/content/thumbs"


@app.get("/training/dataset/{name}/thumb/{file}")
def thumb(name: str, file: str):
    file = clean_file(file)
    src = os.path.join(images_dir(name), file)
    if not os.path.exists(src):
        fail("Image not found", 404)
    key = hashlib.sha1(f"{src}|{os.path.getmtime(src)}".encode()).hexdigest()
    path = os.path.join(THUMBS, key + ".jpg")
    if not os.path.exists(path):
        os.makedirs(THUMBS, exist_ok=True)
        im = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
        im.thumbnail((360, 360))
        im.save(path, "JPEG", quality=85)
    return Response(open(path, "rb").read(), media_type="image/jpeg", headers={"Cache-Control": "max-age=3600"})


def _thumb_response(src):
    key = hashlib.sha1(f"{src}|{os.path.getmtime(src)}".encode()).hexdigest()
    path = os.path.join(THUMBS, key + ".jpg")
    if not os.path.exists(path):
        os.makedirs(THUMBS, exist_ok=True)
        im = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
        im.thumbnail((360, 360))
        im.save(path, "JPEG", quality=85)
    return Response(open(path, "rb").read(), media_type="image/jpeg", headers={"Cache-Control": "max-age=3600"})


class Kind(BaseModel):
    kind: str  # "character" or "edit"


@app.post("/training/dataset/{name}/kind")
def kind(name: str, req: Kind):
    try:
        set_kind(name, req.kind)
    except ValueError as e:
        fail(e)
    return dataset_payload(name)


class ControlUpload(BaseModel):
    file: str      # the training ("after") picture this 'before' picture belongs to
    filename: str
    data: str      # base64 (data: URL prefix allowed)


@app.post("/training/dataset/{name}/control")
def upload_control(name: str, req: ControlUpload):
    target = clean_file(req.file)
    if target not in list_images(name):
        fail("Upload the training picture first", 404)
    ext = os.path.splitext(clean_file(req.filename))[1].lower()
    if ext not in IMAGE_EXTS:
        fail(f"{req.filename}: only jpg, png, webp and bmp images are supported")
    raw = base64.b64decode(re.sub(r"^data:[^,]*,", "", req.data))
    try:
        Image.open(io.BytesIO(raw)).verify()
    except Exception:
        fail(f"{req.filename} is not a readable image")
    os.makedirs(control_dir(name), exist_ok=True)
    old = control_for(name, target)
    if old:
        os.remove(os.path.join(control_dir(name), old))
    with open(os.path.join(control_dir(name), os.path.splitext(target)[0] + ext), "wb") as fh:
        fh.write(raw)
    return dataset_payload(name)


@app.delete("/training/dataset/{name}/control/{file}")
def delete_control(name: str, file: str):
    c = control_for(name, file)
    if c:
        os.remove(os.path.join(control_dir(name), c))
    return dataset_payload(name)


@app.get("/training/dataset/{name}/control_thumb/{file}")
def control_thumb(name: str, file: str):
    c = control_for(name, file)
    if not c:
        fail("No before picture", 404)
    return _thumb_response(os.path.join(control_dir(name), c))


@app.get("/training/dataset/{name}/control_file/{file}")
def control_file(name: str, file: str):
    """The original 'before' picture (for the website's cloud backup)."""
    c = control_for(name, file)
    if not c:
        fail("No before picture", 404)
    ext = os.path.splitext(c)[1].lower().lstrip(".")
    media = {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png", "webp": "image/webp", "bmp": "image/bmp"}.get(ext, "application/octet-stream")
    return Response(open(os.path.join(control_dir(name), c), "rb").read(), media_type=media)


class AutoTag(BaseModel):
    threshold: float = 0.35
    mode: str = "replace"  # "append" to existing captions or "replace" them
    trigger_word: str = ""
    exclude_tags: str = ""  # comma-separated tags to leave out


@app.post("/training/dataset/{name}/autotag")
def autotag(name: str, req: AutoTag):
    try:
        start_autotag(name, req.threshold, req.mode, req.trigger_word, req.exclude_tags)
    except Exception as e:
        fail(e)
    return status_payload()


class Start(BaseModel):
    name: str
    settings: dict = {}


@app.post("/training/start")
def start(req: Start):
    try:
        start_training(req.name, req.settings)
    except Exception as e:
        fail(e)
    return status_payload()


@app.post("/training/dataset/{name}/start_fresh")
def fresh(name: str):
    try:
        result = start_fresh(name)
    except Exception as e:
        fail(e)
    return {**result, **dataset_payload(name)}


@app.post("/training/stop")
def stop():
    stop_task()
    return status_payload()


_tunnel = globals().get("TRAINING_TUNNEL")
if globals().get("TRAINING_SERVER_URL") and _tunnel is not None and _tunnel.poll() is None:
    print(f"The server is already running: {TRAINING_SERVER_URL}")
else:
    if globals().get("TRAINING_SERVER_URL"):
        print("The old tunnel stopped, so the old URL no longer works — starting a new one (new URL below).")
    if not globals().get("TRAINING_UVICORN_STARTED"):
        threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
                         daemon=True).start()
        TRAINING_UVICORN_STARTED = True
        time.sleep(4)
    cf = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                          start_new_session=True)  # survives stopping a notebook cell
    TRAINING_TUNNEL = cf
    url, t0 = None, time.time()
    while time.time() - t0 < 60:
        line = cf.stdout.readline()
        if not line:
            break
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break
    threading.Thread(target=lambda: [None for _ in cf.stdout], daemon=True).start()  # keep the pipe drained
    if url:
        TRAINING_SERVER_URL = url
        print("=" * 60)
        print(f"READY: {url}")
        print("Paste this into the website: LoRA Training tab → Training server URL (it switches to Qwen 2511 training).")
        print("Keep this Colab tab open while you work. Progress is also printed by cell 4 if you want it here.")
        print("=" * 60)
    else:
        print("Could not start the Cloudflare tunnel — run this cell again.")

In [ ]:
# Cell 3 (optional): train WITHOUT the website — fill in the form and run. Skip this if you use the website.
#@markdown Pictures: `MyDrive/ComfyUI/training_qwen/<name>/images` (a file picker opens if the folder is empty).
#@markdown Character LoRAs only here; edit-effect LoRAs (with Before pictures) are easiest on the website.
LORA_NAME = "my_character"  #@param {type:"string"}
TRIGGER_WORD = "zxq_mia"  #@param {type:"string"}
CLASS_WORD = "woman"  #@param {type:"string"}
MAX_TRAIN_STEPS = 1000  #@param {type:"integer"}
NETWORK_DIM = 16  #@param {type:"integer"}
LEARNING_RATE = 1e-4  #@param {type:"number"}

if "clean_name" not in globals():
    raise RuntimeError("Run cell 1 (setup) first, then run this cell again.")

import os, io, zipfile, time
from google.colab import files

name = clean_name(LORA_NAME)
d = images_dir(name)
os.makedirs(d, exist_ok=True)
set_kind(name, "character")
if not list_images(name):
    print(f"Choose images, caption .txt files or one .zip. They'll be saved to {d}")
    for fname, data in files.upload().items():
        if fname.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                for m in z.namelist():
                    base = os.path.basename(m)
                    if base and not base.startswith(".") and "__MACOSX" not in m and base.lower().endswith(IMAGE_EXTS + (".txt",)):
                        open(os.path.join(d, clean_file(base)), "wb").write(z.read(m))
        elif fname.lower().endswith(IMAGE_EXTS + (".txt",)):
            open(os.path.join(d, clean_file(fname)), "wb").write(data)
        if os.path.exists(fname):
            os.remove(fname)
print(f"{len(list_images(name))} images in {d}")

start_training(name, {"trigger_word": TRIGGER_WORD, "class_word": CLASS_WORD, "max_train_steps": MAX_TRAIN_STEPS,
                      "network_dim": NETWORK_DIM, "network_alpha": NETWORK_DIM, "learning_rate": LEARNING_RATE})
last = None
while STATE["status"] == "running":
    msg = f"{STATE['phase']}" + (f" — step {STATE['step']}/{STATE['total']}, {STATE['eta']} left" if STATE["step"] else "")
    if msg != last:
        print(time.strftime("%H:%M"), msg)
        last = msg
    time.sleep(20)
print(STATE["status"], STATE.get("result") or STATE.get("error") or "")
if STATE["status"] == "done":
    print(f"Start a Qwen edit notebook and pick {STATE['result']['lora']} in the website's LoRA list; "
          f"put a picture of the character as Picture 1 and use '{TRIGGER_WORD}' in the prompt.")

In [ ]:
# Cell 4 (optional): show the current task's progress and log here in Colab
if "clean_name" not in globals():
    raise RuntimeError("Run cell 1 (setup) first, then run this cell again.")

import time
from IPython.display import clear_output
try:
    while True:
        s = status_payload()
        clear_output(wait=True)
        print(f"{s['task'] or 'nothing running'} {s['name'] or ''}: {s['status']} — {s['phase']}")
        if s["total"]:
            print(f"step {s['step']}/{s['total']}  elapsed {s['elapsed']}  left {s['eta']}")
        print("\n".join(s["log"][-15:]))
        if s["status"] != "running":
            break
        time.sleep(10)
except KeyboardInterrupt:
    pass